# S&P500/VIX Promoted Discrete Benchmark

Report-facing notebook for the promoted public method: a standard causal VQ tokenizer followed by an additive scalar-conditioned causal autoregressive token prior. The continuous TC-VAE checkpoint is treated only as an optional reference, and RVQ q2 is kept as an ablation. This notebook does not introduce diffusion, AdaLN, cross-attention, GroupedRVQ, or MGVQ.


## Parameters

Paths default to public-repository artefacts under ignored `outputs/` directories. Leave `RUN_PAPER_STYLE_EVALUATION=False` when using existing generated summaries; set it to `True` only after the tokenizer, token artefacts, trained additive prior, and optional continuous reference checkpoint exist locally.


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import yaml
from IPython.display import Image, Markdown, display

RUN_PAPER_STYLE_EVALUATION = False
SEED = 99
N_SAMPLE = 1000
TEMPERATURE = 1.0
TOP_K = 40

TOKENIZER_CONFIG = Path("configs/experiments/sp500_vix_causal_vq_tokenizer.yaml")
TOKEN_PRIOR_CONFIG = Path("configs/experiments/sp500_vix_causal_token_prior_additive.yaml")
CONTINUOUS_CONFIG = Path("configs/experiments/sp500_vix_beta_cvae.yaml")

STANDARD_TOKENIZER_DIR = Path("outputs/sp500_vix_discrete/tokenizer/sp500_vix_causal_vq_tokenizer_seed0")
STANDARD_TOKEN_DATA_DIR = Path("outputs/sp500_vix_discrete/token_prior/tokens_codebook64_codebookdim16")
STANDARD_GEOMETRY_DIR = Path("outputs/latent_geometry/sp500_vix_standard_vq")
RVQ_Q2_GEOMETRY_DIR = Path("outputs/latent_geometry/sp500_vix_rvq_q2")

# Replace this placeholder after training the additive prior.
DISCRETE_PRIOR_DIR = Path("outputs/sp500_vix_discrete/token_prior/additive/<prior-run>")
# Optional continuous reference. The paper-style script skips it when unavailable.
CONTINUOUS_MODEL_DIR = Path("outputs/full/sp500_vix_beta_cvae/<training-run>/final_model")
PAPER_STYLE_DIR = Path("outputs/sp500_vix_discrete/paper_style")
BASE_DATA_DIR = Path("data/processed")


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


REPO_ROOT = find_repo_root()


def repo_path(path: Path) -> Path:
    return path if path.is_absolute() else (REPO_ROOT / path).resolve()


def display_path(path: Path) -> str:
    resolved = repo_path(path)
    try:
        return str(resolved.relative_to(REPO_ROOT))
    except ValueError:
        return str(resolved)


def load_json(path: Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    return json.loads(resolved.read_text())

print(f"Repository root: {REPO_ROOT}")
print(f"RUN_PAPER_STYLE_EVALUATION: {RUN_PAPER_STYLE_EVALUATION}")


## Method Configuration

The promoted configuration uses one token per time step, scalar VIX conditioning, and an additive conditional causal token prior. The table is a quick guardrail before generating report figures.


In [ ]:
def flatten_config_rows(name: str, path: Path) -> list[dict[str, Any]]:
    resolved = repo_path(path)
    if not resolved.exists():
        return [{"config": name, "field": "missing", "value": display_path(path)}]
    raw = yaml.safe_load(resolved.read_text()) or {}
    rows: list[dict[str, Any]] = []
    for section in ["dataset", "tokenizer", "model", "token_prior", "training"]:
        value = raw.get(section)
        if isinstance(value, dict):
            for key, item in value.items():
                if key in {"diffusion", "adaln", "cross_attention", "grouped_rvq", "mgvq"}:
                    continue
                if isinstance(item, (str, int, float, bool)) or item is None:
                    rows.append({"config": name, "field": f"{section}.{key}", "value": item})
    return rows

rows = []
rows.extend(flatten_config_rows("standard VQ tokenizer", TOKENIZER_CONFIG))
rows.extend(flatten_config_rows("additive token prior", TOKEN_PRIOR_CONFIG))
rows.extend(flatten_config_rows("continuous reference", CONTINUOUS_CONFIG))
config_table = pd.DataFrame(rows)
display(config_table)

for forbidden in ["diffusion", "adaln", "cross_attention", "grouped_rvq", "mgvq"]:
    matches = config_table[config_table["value"].astype(str).str.lower().str.contains(forbidden, regex=False)]
    if not matches.empty:
        display(Markdown(f"**Check `{forbidden}` manually:**"))
        display(matches)


## Artefact Inventory

The final report should distinguish completed latent-geometry evidence from paper-style generation metrics that still require a trained additive prior.


In [ ]:
required_paths = {
    "tokenizer config": TOKENIZER_CONFIG,
    "token prior config": TOKEN_PRIOR_CONFIG,
    "standard tokenizer dir": STANDARD_TOKENIZER_DIR,
    "standard token data dir": STANDARD_TOKEN_DATA_DIR,
    "standard geometry dir": STANDARD_GEOMETRY_DIR,
    "base data dir": BASE_DATA_DIR,
}
optional_paths = {
    "trained additive prior dir": DISCRETE_PRIOR_DIR,
    "continuous reference model dir": CONTINUOUS_MODEL_DIR,
    "paper-style output dir": PAPER_STYLE_DIR,
    "RVQ q2 geometry dir": RVQ_Q2_GEOMETRY_DIR,
}

rows = []
for name, path in required_paths.items():
    resolved = repo_path(path)
    rows.append({"role": "required", "name": name, "path": display_path(path), "exists": resolved.exists()})
for name, path in optional_paths.items():
    resolved = repo_path(path)
    rows.append({"role": "optional", "name": name, "path": display_path(path), "exists": resolved.exists()})
inventory = pd.DataFrame(rows)
display(inventory)

missing_required = inventory[(inventory["role"] == "required") & (~inventory["exists"])]
if not missing_required.empty:
    display(Markdown("**Missing required artefacts for the promoted notebook:**"))
    display(missing_required)
else:
    print("All required tokenizer and geometry artefacts are present.")


## Latent-Geometry Decision Table

These numbers are the report-ready standard-VQ evidence already regenerated from public-repository artefacts. RVQ q2 is shown only to document the ablation and the multi-code sampling difficulty.


In [ ]:
standard_summary = load_json(STANDARD_GEOMETRY_DIR / "codebook_geometry_summary.json") or {}
rvq_summary = load_json(RVQ_Q2_GEOMETRY_DIR / "codebook_geometry_summary.json") or {}
rvq_pair_summary = load_json(RVQ_Q2_GEOMETRY_DIR / "q0_q1_pair_summary.json") or {}


def geometry_row(name: str, summary: dict[str, Any]) -> dict[str, Any]:
    usage = summary.get("usage", {})
    geometry = summary.get("geometry", {})
    metadata = summary.get("metadata", {})
    return {
        "setting": name,
        "quantizer": metadata.get("quantizer_type"),
        "embedding_shape": geometry.get("embedding_shape"),
        "active_codes": usage.get("active_code_count"),
        "perplexity": usage.get("codebook_perplexity"),
        "entropy": usage.get("entropy"),
        "projection": geometry.get("projection_method"),
    }

summary_rows = [geometry_row("standard VQ", standard_summary)]
if rvq_summary:
    summary_rows.append(geometry_row("RVQ q2 ablation", rvq_summary))
display(pd.DataFrame(summary_rows))

if standard_summary:
    vix_rows = standard_summary.get("condition_buckets", [])
    if vix_rows:
        display(Markdown("### Standard VQ VIX buckets"))
        display(pd.DataFrame(vix_rows))

if rvq_summary.get("usage", {}).get("per_quantizer"):
    display(Markdown("### RVQ q2 per-quantizer usage"))
    display(pd.DataFrame(rvq_summary["usage"]["per_quantizer"]))

if rvq_pair_summary:
    display(Markdown("### RVQ q2 same-time q0/q1 support"))
    pair_fields = [
        "pair_count",
        "active_pair_count",
        "active_pair_ratio",
        "absent_pair_mass",
        "zero_count_pairs",
        "rare_active_pairs_1_to_5",
    ]
    display(pd.DataFrame([{field: rvq_pair_summary.get(field) for field in pair_fields}]))


## Report Figures

The first group is the promoted standard-VQ figure set. The RVQ q2 heatmap is suitable only for the ablation paragraph explaining sparse same-time q0/q1 support. Displayed images are not embedded once notebook outputs are stripped.


In [ ]:
figure_manifest = [
    {
        "role": "primary",
        "path": STANDARD_GEOMETRY_DIR / "codebook_projection.png",
        "caption": "PCA projection of the standard VQ codebook used by the promoted tokenizer.",
    },
    {
        "role": "primary",
        "path": STANDARD_GEOMETRY_DIR / "codebook_usage_projection.png",
        "caption": "Standard VQ codebook projection with empirical usage overlay.",
    },
    {
        "role": "primary",
        "path": STANDARD_GEOMETRY_DIR / "vix_bucket_code_usage.png",
        "caption": "VIX-bucket code usage showing broader code utilisation in higher-volatility windows.",
    },
    {
        "role": "primary",
        "path": STANDARD_GEOMETRY_DIR / "token_trajectory_examples.png",
        "caption": "Example standard-VQ token trajectories through time.",
    },
    {
        "role": "optional",
        "path": STANDARD_GEOMETRY_DIR / "codebook_voronoi.png",
        "caption": "Two-dimensional Voronoi view of nearest projected standard-VQ codebook regions.",
    },
    {
        "role": "ablation",
        "path": RVQ_Q2_GEOMETRY_DIR / "q0_q1_pair_heatmap.png",
        "caption": "RVQ q2 same-time q0/q1 pair heatmap showing sparse joint support.",
    },
]
manifest_df = pd.DataFrame(
    [
        {
            "role": item["role"],
            "path": display_path(item["path"]),
            "exists": repo_path(item["path"]).exists(),
            "caption": item["caption"],
        }
        for item in figure_manifest
    ]
)
display(manifest_df)

for item in figure_manifest:
    resolved = repo_path(item["path"])
    if resolved.exists():
        display(Markdown(f"### {item['role']}: `{display_path(item['path'])}`"))
        display(Markdown(item["caption"]))
        display(Image(filename=str(resolved)))


## Optional Paper-Style Evaluation

This command samples the additive scalar-conditioned causal AR prior, decodes through the frozen standard VQ tokenizer, and writes report figures. It should be run only after replacing `DISCRETE_PRIOR_DIR` with a real trained-prior directory.


In [ ]:
paper_style_command = [
    sys.executable,
    "scripts/evaluate_sp500_vix_paper_style.py",
    "--discrete-config",
    display_path(TOKEN_PRIOR_CONFIG),
    "--discrete-prior-dir",
    display_path(DISCRETE_PRIOR_DIR),
    "--discrete-tokenizer-dir",
    display_path(STANDARD_TOKENIZER_DIR),
    "--continuous-config",
    display_path(CONTINUOUS_CONFIG),
    "--continuous-model-dir",
    display_path(CONTINUOUS_MODEL_DIR),
    "--output-dir",
    display_path(PAPER_STYLE_DIR),
    "--base-data-dir",
    display_path(BASE_DATA_DIR),
    "--n-sample",
    str(N_SAMPLE),
    "--seed",
    str(SEED),
    "--temperature",
    str(TEMPERATURE),
    "--top-k",
    str(TOP_K),
]
print("Equivalent paper-style command:")
print(" ".join(shlex.quote(part) for part in paper_style_command))

prior_ready = repo_path(DISCRETE_PRIOR_DIR).exists() and "<" not in str(DISCRETE_PRIOR_DIR)
if RUN_PAPER_STYLE_EVALUATION:
    if not prior_ready:
        print("DISCRETE_PRIOR_DIR is still a placeholder or does not exist; train the additive prior first.")
    else:
        subprocess.run(paper_style_command, cwd=REPO_ROOT, check=True)
else:
    print("RUN_PAPER_STYLE_EVALUATION=False; no sampling run was launched.")


## Paper-Style Result Loader

Once `scripts/evaluate_sp500_vix_paper_style.py` has been run, this cell loads numeric summaries and lists generated figures ready for report inclusion.


In [ ]:
paper_summary = load_json(PAPER_STYLE_DIR / "paper_style_summary.json")
if paper_summary is None:
    print(f"No paper-style summary found at {display_path(PAPER_STYLE_DIR / 'paper_style_summary.json')}.")
    print("Do not report generation metrics until this file exists for the final additive prior run.")
else:
    comparisons = paper_summary.get("comparisons", {})
    if comparisons:
        display(pd.DataFrame.from_dict(comparisons, orient="index"))
    token_diagnostics = paper_summary.get("token_diagnostics")
    if isinstance(token_diagnostics, dict):
        display(Markdown("### Token diagnostics"))
        display(pd.DataFrame([token_diagnostics]))

    generated = sorted(repo_path(PAPER_STYLE_DIR).glob("*.png"))
    display(pd.DataFrame({"figure": [display_path(path) for path in generated]}))


## Report Interpretation

The standard VQ tokenizer is the promoted representation because it uses 63 of 64 codes, has global perplexity about 39 on the S&P500/VIX token artefacts, and shows increasing code utilisation across VIX buckets. This supports a scalar-conditioned discrete representation without adding a multi-code tokenizer.

The RVQ q2 result should be described as an ablation. It gives an interpretable coarse/detail split, but only 231 of 4096 same-time q0/q1 pairs are active, so the prior must learn sparse joint support as well as temporal and VIX-dependent dynamics. That is the reason to keep standard VQ as the report baseline and leave GroupedRVQ, MGVQ, diffusion, AdaLN, and cross-attention out of the final method.
